In [15]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

In [16]:
data = pd.read_csv("C://Users//DELL//PycharmProjects//PythonProject_practice_2025//Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [17]:
# Preprocess the data
data = data.drop(['RowNumber','CustomerId','Surname'], axis = 1)

In [18]:
# Encode Categorical variables

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [19]:

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')

geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()

geo_encoder_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

geo_encoder_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [20]:
data = pd.concat([data.drop('Geography', axis=1), geo_encoder_df], axis =1)
data.head(10)

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
5,645,1,44,8,113755.78,2,1,0,149756.71,1,0.0,0.0,1.0
6,822,1,50,7,0.00,2,1,1,10062.80,0,1.0,0.0,0.0
7,376,0,29,4,115046.74,4,1,0,119346.88,1,0.0,1.0,0.0
8,501,1,44,4,142051.07,2,0,1,74940.50,0,1.0,0.0,0.0
9,684,1,27,2,134603.88,1,1,1,71725.73,0,1.0,0.0,0.0


In [21]:
# split data into features and target
X = data.drop('EstimatedSalary', axis = 1)
y = data['EstimatedSalary']

In [22]:
# save the encoder and scaler for later use
with open('label_encoder_gender_reg.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo_reg.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)



In [23]:
## split the data in training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [24]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [25]:
with open('scaler_reg.pkl', 'wb') as file:
    pickle.dump(scaler, file)

ANN Regression problem

In [26]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import datetime


In [27]:
model = Sequential([
    Dense(64, activation = 'relu', input_shape = (X_train.shape[1],)), ## HL1 connected with input layer
    Dense(32, activation = 'relu'), ## HL2
    Dense(1) ## output layer - when we don't apply any activation function the default activation function will be linear activation function
])

## compile the model

model.compile(optimizer = "adam", loss = 'mean_absolute_error', metrics=['mae'])

model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [28]:
## setup the Tensorboard

from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq = 1)

In [29]:
## setup early stopping

early_stopping_callback = EarlyStopping(monitor = 'val_loss', patience = 10, restore_best_weights = True)

In [30]:
## Training the model

history = model.fit(
    X_train, y_train, validation_data = (X_test, y_test), epochs = 100,
    callbacks = [tensorflow_callback, early_stopping_callback]
)

Epoch 1/100


250/250 [==============================] - 4s 7ms/step - loss: 100380.8750 - mae: 100380.8750 - val_loss: 98524.1094 - val_mae: 98524.1094
Epoch 2/100
250/250 [==============================] - 2s 6ms/step - loss: 99648.3672 - mae: 99648.3672 - val_loss: 97031.6797 - val_mae: 97031.6797
Epoch 3/100
250/250 [==============================] - 1s 5ms/step - loss: 97067.7891 - mae: 97067.7891 - val_loss: 93225.3125 - val_mae: 93225.3125
Epoch 4/100
250/250 [==============================] - 2s 9ms/step - loss: 91980.2500 - mae: 91980.2500 - val_loss: 86845.8906 - val_mae: 86845.8906
Epoch 5/100
250/250 [==============================] - 1s 5ms/step - loss: 84535.3984 - mae: 84535.3984 - val_loss: 78587.4609 - val_mae: 78587.4609
Epoch 6/100
250/250 [==============================] - 1s 6ms/step - loss: 75743.3359 - mae: 75743.3359 - val_loss: 69964.4453 - val_mae: 69964.4453
Epoch 7/100
250/250 [==============================] - 1s 5ms/step - loss: 67081.3359 - mae: 67081.335

In [31]:
## load tensorboard extention

%load_ext tensorboard

In [32]:
%tensorboard --logdir regressionlogs/fit

In [34]:
## Evaluate model on the test data

test_loss, test_mae = model.evaluate(X_test, y_test)
print(f'Test Mae : {test_mae}')
print(f'Test loss: {test_loss}')

63/63 [==============================] - 0s 5ms/step - loss: 50319.6133 - mae: 50319.6133
Test Mae : 50319.61328125
Test loss: 50319.61328125


In [35]:
model.save('regression_model.h5')

C:\Users\DELL\PyCharmMiscProject\.venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
